# Notebook 04 — Deployment: how a fine-tuned model actually serves

**Workshop:** Agentic AI for Scientists — Week 4 (Post-Training & Deployment)
**Block:** deployment (20 min)
**Goal:** A trained model on disk is not a product. This notebook covers the **serving** half of "Post-Training & Deployment": load your tuned model, generate, and walk the real options — local, managed endpoint, on-device — and what each trades off in **latency, cost, and privacy** (the last one matters a lot for clinical data).

```
   train (NB01-03)  ->  a model on disk  ->  SERVE IT  ->  something that answers requests
                                              ^ you are here
```

## 0. Setup & load the tuned model

In [ ]:
%pip install -q unsloth transformers accelerate

In [ ]:
from unsloth import FastLanguageModel
import torch, os
from pathlib import Path

SYSTEM_PROMPT = (
    "You are a clinical assistant. Read the medical question and respond with a JSON object "
    "containing: disease, patient_info, symptoms (list), treatment (list), answer_summary."
)

# Prefer a model you trained in NB01-03; fall back to the base model so this notebook
# always runs standalone.
CANDIDATES = ["qwen3-medquad-qlora-merged", "qwen3-medquad-full-sft", "Qwen/Qwen3-0.6B"]
MODEL_PATH = next((c for c in CANDIDATES if c.startswith("Qwen/") or Path(c).exists()), "Qwen/Qwen3-0.6B")
print(f"Loading: {MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH, max_seq_length=1024, dtype=None, load_in_4bit=False)
FastLanguageModel.for_inference(model)

---
## 1. Pattern A — local in-process inference

The simplest deployment: load the model in your process and call `generate()`. Zero network, full privacy, but you hold the GPU/RAM and there's no batching/queueing. Great for notebooks, research scripts, and an Organon **skill** running on your own machine (that's the Week 6 capstone).

In [ ]:
import time

def generate(question: str, max_new_tokens=200, temperature=0.3):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    t0 = time.time()
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=temperature > 0)
    dt = time.time() - t0
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    n_new = out.shape[1] - ids.shape[1]
    print(f"[{dt:.2f}s, {n_new} tok, {n_new/dt:.1f} tok/s]\n{text}")
    return text

_ = generate("What are the warning signs of a stroke and what should be done immediately?")

---
## 2. Pattern B — a managed inference endpoint (the cloud path)

For a service many people call, you don't hand-roll `generate()`. You put the model behind a server that does **continuous batching**, queueing, and an OpenAI-compatible API — e.g. **Hugging Face Inference Endpoints** (managed **TGI**) or **vLLM** self-hosted. Your client code becomes provider-agnostic:

```python
from openai import OpenAI
client = OpenAI(base_url="https://<your-endpoint>/v1/", api_key=os.environ["HF_TOKEN"])
resp = client.chat.completions.create(
    model="qwen3-medquad", messages=[{"role": "user", "content": question}], max_tokens=200)
print(resp.choices[0].message.content)
```

You push the merged model to the Hub (NB03), click "Deploy -> Inference Endpoint", pick a GPU, and you get that URL. We don't spin one up live (it bills by the hour), but the snippet is exactly what you'd run. **vLLM** equivalent: `vllm serve qwen3-medquad --port 8000` then point the same OpenAI client at `http://localhost:8000/v1`.

In [ ]:
# Illustrative only (not executed): the serving server replaces your local generate().
SERVING_SNIPPET = """
# Self-host with vLLM (one command, OpenAI-compatible server):
#   pip install vllm
#   vllm serve YOUR_USERNAME/qwen3-medquad-qlora --port 8000
#
# Then from anywhere:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
r = client.chat.completions.create(
    model="YOUR_USERNAME/qwen3-medquad-qlora",
    messages=[{"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user", "content": "Describe the symptoms of anemia."}],
    max_tokens=200, temperature=0.3)
print(r.choices[0].message.content)
"""
print(SERVING_SNIPPET)

---
## 3. Pattern C — on-device (GGUF + llama.cpp / Ollama)

For privacy-critical or offline use (a laptop in a clinic with no cloud), you convert the model to **GGUF** and run it on CPU/Metal with **llama.cpp** or **Ollama** — no GPU, no network, data never leaves the device. Unsloth can export GGUF directly:

```python
model.save_pretrained_gguf("qwen3-medquad-gguf", tokenizer, quantization_method="q4_k_m")
# then:  ollama create qwen3-medquad -f Modelfile   &&   ollama run qwen3-medquad
```

A 0.6B model in `q4_k_m` is ~400 MB and runs comfortably on a modern laptop. This is the deployment that makes the **Week 6 Organon skill** practical: a local model your agent can call all day for free.

---
## 4. Choosing a deployment — the trade-off table

| Pattern | Latency | Cost | Privacy | Scales to many users | Best for |
|---|---|---|---|---|---|
| **A. Local in-process** | low (1 user) | free (your hardware) | total | no | notebooks, research scripts, an Organon skill |
| **B. Managed endpoint / vLLM** | low + batched | $/hour while up | data leaves your box | yes | a shared service, an app backend |
| **C. On-device GGUF** | medium (CPU) | free | total, offline | no | clinics, field work, privacy-critical |

**For clinical / scientific data, privacy often decides this before latency or cost does.** A small fine-tuned model you can run on-device (C) or in-process (A) is frequently the *right* answer even when a giant hosted model would score higher on a benchmark.

---
## Where this goes next

- **Week 5 — Evaluation & Benchmarking.** We have models and we can serve them. But are they *good*? Week 5 measures the NB01-03 models against held-out MedQuAD ground truth (entity-level F1), builds an LLM-as-judge, and runs a published benchmark. "Did post-training work?" becomes a number.
- **Week 6 — Towards Full Autonomy.** We take the deployed model (Pattern A/C) and wire it into **Organon as a skill** that runs a daily clinical-research chore on its own — closing the loop back to Week 1.

You now have the full Post-Training & Deployment arc: **prepare data -> fine-tune (full / LoRA / QLoRA) -> serve**.